In [1]:
import pandas as pd

In [2]:
rollNumber="1024170051"
lastTwoDigits=rollNumber[-2:]
digit1=int(lastTwoDigits[0])
digit2=int(lastTwoDigits[1])
categories=["billing","account","general"]
category1=categories[digit1%3]
category2=categories[digit2%3]
print("Last two digits:",lastTwoDigits)
print("Digit",digit1,"category:",category1)
print("Digit",digit2,"category:",category2)

Last two digits: 51
Digit 5 category: general
Digit 1 category: account


In [3]:
fixed_entries=[
    {
        "question":"what is the annual fee",
        "answer":"The annual fee is Rs 500.",
        "keywords":"fee cost price charge",
        "category":"billing"
    },
    {
        "question":"how to reset password",
        "answer":"Go to Settings>Reset Password.",
        "keywords":"password reset login",
        "category":"account"
    },
    {
        "question":"what are your working hours",
        "answer":"We are open 9 AM to 5 PM.",
        "keywords":"hours timing open time",
        "category":"general"
    },
    {
        "question":"how can i pay the fee",
        "answer":"You can pay via UPI, card, or net banking.",
        "keywords":"pay payment upi fee",
        "category":"billing"
    }
]

In [4]:
def create_entry(category,digit):
    if category=="billing":
        return{
            "question":f"How can I check how much I need to pay?",
            "answer":"You can check your fee details in the billing section.",
            "keywords":"fee payment amount",
            "category":category
        }

    elif category=="account":
        return{
            "question":f"How can I change my account details?",
            "answer":"You can update your account details from the account settings.",
            "keywords":"account update profile",
            "category":category
        }

    else:
        return{
            "question":f"What services do you provide?",
            "answer":"You can view all available services from the main menu.",
            "keywords":"services help information",
            "category":category
        }
entry1 = create_entry(category1,digit1)
entry2 = create_entry(category2,digit2)
personalized_entries=[
    entry1,entry2
]

In [5]:
allEntries=fixed_entries+personalized_entries
df=pd.DataFrame(allEntries)
print(df)

                               question  \
0                what is the annual fee   
1                 how to reset password   
2           what are your working hours   
3                 how can i pay the fee   
4         What services do you provide?   
5  How can I change my account details?   

                                              answer  \
0                          The annual fee is Rs 500.   
1                     Go to Settings>Reset Password.   
2                          We are open 9 AM to 5 PM.   
3         You can pay via UPI, card, or net banking.   
4  You can view all available services from the m...   
5  You can update your account details from the a...   

                    keywords category  
0      fee cost price charge  billing  
1       password reset login  account  
2     hours timing open time  general  
3        pay payment upi fee  billing  
4  services help information  general  
5     account update profile  account  


In [6]:
def scoreQuery(query,df):
    queryWords=set(query.lower().split())
    results=[]
    for index,row in df.iterrows():
        questionWords = set(row["question"].lower().split())
        keywordWords = set(row["keywords"].lower().split())
        allWords = questionWords | keywordWords
        matchedWords = queryWords & allWords
        score = len(matchedWords)
        if score > 0:
            confidence = score / len(queryWords)
            results.append({
                "index": index,
                "question": row["question"],
                "answer": row["answer"],
                "category": row["category"],
                "score": score,
                "confidence": round(confidence, 2)
            })
    results.sort(key=lambda x: x["confidence"], reverse=True)
    return results

In [7]:
query=input("Enter your query:")
results=scoreQuery(query, df)
for result in results:
    print(result)

Enter your query: how do i update my mobile number


{'index': 5, 'question': 'How can I change my account details?', 'answer': 'You can update your account details from the account settings.', 'category': 'account', 'score': 4, 'confidence': 0.57}
{'index': 3, 'question': 'how can i pay the fee', 'answer': 'You can pay via UPI, card, or net banking.', 'category': 'billing', 'score': 2, 'confidence': 0.29}
{'index': 1, 'question': 'how to reset password', 'answer': 'Go to Settings>Reset Password.', 'category': 'account', 'score': 1, 'confidence': 0.14}
{'index': 4, 'question': 'What services do you provide?', 'answer': 'You can view all available services from the main menu.', 'category': 'general', 'score': 1, 'confidence': 0.14}


In [8]:
def sameCategory(categoryName):
    return df[
        df["category"].str.lower() == categoryName.lower()
    ]["question"].tolist()

In [9]:
category=input("Enter category:")
print(sameCategory(category))

Enter category: account


['how to reset password', 'How can I change my account details?']


In [10]:
df

,question,answer,keywords,category
0,what is the annual fee,The annual fee is Rs 500.,fee cost price charge,billing
1,how to reset password,Go to Settings>Reset Password.,password reset login,account
2,what are your working hours,We are open 9 AM to 5 PM.,hours timing open time,general
3,how can i pay the fee,"You can pay via UPI, card, or net banking.",pay payment upi fee,billing
4,What services do you provide?,You can view all available services from the m...,services help information,general
5,How can I change my account details?,You can update your account details from the a...,account update profile,account


In [11]:
index=int(input("Enter the index of the FAQ entry to modify:"))
newKeyword=input("Enter a new keyword:")
df.at[index,"keywords"]=df.at[index,"keywords"]+" "+ newKeyword
print("Updated entry:")
print(df.loc[index])

Enter the index of the FAQ entry to modify: 1
Enter a new keyword: security


Updated entry:
question             how to reset password
answer      Go to Settings>Reset Password.
keywords     password reset login security
category                           account
Name: 1, dtype: object


In [14]:
filename="1024170051_faq_data.csv"
df.to_csv(filename,index=False)
print("File saved as:",filename)

File saved as: 1024170051_faq_data.csv


In [15]:
categoryCounts=df.groupby("category").size()
print(categoryCounts)

category
account    2
billing    2
general    2
dtype: int64


In [18]:
def findBestMatches(query, df):
    results=scoreQuery(query, df)
    if len(results)==0:
        print("Sorry, I could not find any matching FAQ.")
        return
    highestScore=results[0]["score"]
    bestMatches = []
    for result in results:
        if result["score"]==highestScore:
            bestMatches.append(result)

    print("Best matching FAQ(s):")
    for match in bestMatches:
        print("\nQuestion:", match["question"])
        print("Answer:", match["answer"])
        print("Category:", match["category"])
        print("Score:", match["score"])

In [19]:
query="fee"

findBestMatches(query,df)

Best matching FAQ(s):

Question: what is the annual fee
Answer: The annual fee is Rs 500.
Category: billing
Score: 1

Question: how can i pay the fee
Answer: You can pay via UPI, card, or net banking.
Category: billing
Score: 1
